# MPC + GP + BO 통합 유도 파이프라인

## 비례항법을 넘어서 — 최적 유도의 완결

---

> **개인 연구 노트** | Personal Research Note  
> 노트북 07~09에서 개별적으로 다뤘던 MPC, GP, BO를 하나의 파이프라인으로 통합한다.  
> *Integrating MPC, GP residual correction, and Bayesian Optimization into a single guidance pipeline.*

---

## 연구 동기

지금까지 세 가지 기법을 각각 공부했다:

| 노트북 | 주제 | 핵심 아이디어 |
|--------|------|---------------|
| **07** | MPC 유도 | 미래 N스텝을 예측하며 최적 가속도 시퀀스 계산 |
| **08** | GP 공력 보정 | 공칭 모델과 실제 동역학의 불일치를 GP로 학습 |
| **09** | Bayesian Optimization | 최소 평가 횟수로 유도 파라미터 최적 튜닝 |

각각은 독립적으로 의미가 있지만, 진정한 가치는 **통합**에 있다:

$$\boxed{\text{MPC (최적 제어)} + \text{GP (모델 보정)} + \text{BO (가중치 튜닝)} = \text{완결된 유도 파이프라인}}$$

**이 노트북의 목표:**
1. 공칭 모델과 실제 동역학이 다른 상황에서 MPC 유도 성능 평가
2. GP로 모델 불일치를 학습하고 MPC에 주입 → 성능 개선 확인
3. BO로 MPC 가중치를 자동 튜닝 → 최종 성능 극대화
4. APN 기준선 대비 4가지 방법의 정량 비교

```
Pipeline: Data Collection → GP Training → MPC+GP → BO Tuning → Final Evaluation
```

In [34]:
"""Section 1: Setup — 라이브러리 임포트 및 환경 설정"""

import sys
import os
import time
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import rcParams

# 한글 폰트 설정
rcParams['font.family'] = 'Apple SD Gothic Neo'
rcParams['axes.unicode_minus'] = False
rcParams['figure.dpi'] = 110
rcParams['axes.grid'] = True
rcParams['grid.alpha'] = 0.3

# 프로젝트 모듈 임포트
from src.guidance.proportional_navigation import ProportionalNavigation, compute_los_geometry
from src.guidance.mpc_casadi import MPCGuidance
from src.guidance.gp_residual import GPResidualModel
from src.dynamics.los_relative_dynamics import LOSRelativeDynamics
from src.dynamics.missile_3dof import Missile3DOF
from src.targets.target_models import Target

import casadi as ca

np.random.seed(42)

# 결과 저장 디렉토리
os.makedirs('../results', exist_ok=True)

print("임포트 완료")
print(f"  NumPy {np.__version__}")
print(f"  CasADi {ca.__version__}")

임포트 완료
  NumPy 2.0.2
  CasADi 3.7.2


In [35]:
"""True Dynamics Wrapper — 공칭 모델과 다른 '실제' 동역학 정의

공칭 모델 (Missile3DOF 기본값):
    CD = CD_0 + CD_alpha2 * alpha^2
    CL = CL_alpha * alpha
    tau_ap = 0.05s

실제 (True) 동역학:
    CD_true = CD_0 + CD_alpha2 * alpha^2 + CD_alpha4 * alpha^4 + CD_transonic(M)
    CL_true = CL_alpha * alpha - CL_alpha3 * alpha^3
    tau_ap_true = tau_nominal * 1.2  (20% 더 느린 자동조종)
"""


class TrueDynamicsWrapper:
    """공칭 Missile3DOF를 감싸면서 비선형 공력 + 느린 오토파일럿을 적용.

    MPC 솔버는 공칭 모델(LOSRelativeDynamics)을 사용하고,
    실제 시뮬레이션은 이 클래스의 동역학으로 전파한다.
    """

    def __init__(self, base_missile=None):
        self.base = base_missile or Missile3DOF()
        # 비선형 공력 추가 파라미터
        self.CD_alpha4 = 50.0       # alpha^4 항 (고받음각 비선형성)
        self.CL_alpha3 = 5.0        # alpha^3 항 (양력 비선형성)
        self.CD_transonic_amp = 0.15  # 천음속 drag bump 크기
        self.CD_transonic_M0 = 1.0   # 천음속 bump 중심 마하수
        self.CD_transonic_width = 0.15  # 천음속 bump 폭
        # 자동조종 시상수 불확도
        self.tau_scale = 1.2  # 실제 tau = nominal * 1.2

    def get_true_CD(self, alpha_eff, mach):
        """비선형 항력 계수 (True)."""
        CD_base = self.base.CD_0 + self.base.CD_alpha2 * alpha_eff**2
        CD_nl = self.CD_alpha4 * alpha_eff**4
        CD_trans = self.CD_transonic_amp * np.exp(
            -((mach - self.CD_transonic_M0) / self.CD_transonic_width)**2
        )
        return CD_base + CD_nl + CD_trans

    def get_true_CL(self, alpha_eff):
        """비선형 양력 곡선 기울기 (True)."""
        return self.base.CL_alpha * alpha_eff - self.CL_alpha3 * alpha_eff**3

    @property
    def tau_ap_true(self):
        return 0.05 * self.tau_scale  # 기본 tau=0.05 * 1.2 = 0.06


# 인스턴스 생성
missile_nominal = Missile3DOF()
true_dyn = TrueDynamicsWrapper(missile_nominal)

print("동역학 모델 정의 완료")
print(f"  공칭 CD: CD_0={missile_nominal.CD_0}, CD_alpha2={missile_nominal.CD_alpha2}")
print(f"  실제 CD: 공칭 + CD_alpha4={true_dyn.CD_alpha4}*a^4 + transonic bump")
print(f"  공칭 tau_ap: 0.05s")
print(f"  실제 tau_ap: {true_dyn.tau_ap_true:.3f}s (x{true_dyn.tau_scale})")

동역학 모델 정의 완료
  공칭 CD: CD_0=0.35, CD_alpha2=8.0
  실제 CD: 공칭 + CD_alpha4=50.0*a^4 + transonic bump
  공칭 tau_ap: 0.05s
  실제 tau_ap: 0.060s (x1.2)


---
## 2. 교전 시나리오 정의 및 시뮬레이션 엔진

### 표준 교전 시나리오

| 파라미터 | 값 | 설명 |
|----------|-----|------|
| $R_0$ | 10 km | 초기 교전 거리 |
| $V_M$ | 300 m/s | 유도탄 속도 |
| $V_T$ | 200 m/s | 표적 속도 |
| 교전 형태 | Head-on + offset | $\psi_{HE} \approx 180°$, 횡방향 300m 오프셋 |
| 표적 기동 | Weaving | 5g, $\omega = 0.5$ rad/s |
| $\tau_{ap}$ (공칭) | 50 ms | 1차 지연 자동조종 |
| $\tau_{ap}$ (실제) | 60 ms | 20% 불확도 |
| $a_{max}$ | 40g | 가속도 포화 한계 |

### `run_engagement()` 함수

교전 시뮬레이션의 핵심 엔진:
- 입력: `guidance_func(t, r_M, v_M, r_T, v_T, a_pitch_ach, a_yaw_ach) -> (a_pitch_cmd, a_yaw_cmd)`
- NED 좌표계에서 물리 전파 (Missile3DOF 기반)
- LOS 좌표 변환: `LOSRelativeDynamics.ned_to_los_state()`
- 1차 지연 자동조종: $a_{ach} \leftarrow a_{ach} + (1 - e^{-dt/\tau}) \cdot (a_{cmd} - a_{ach})$
- 종료 조건: 거리 증가 (miss detection) 또는 $R < 1$ m

In [36]:
"""교전 시나리오 정의 및 run_engagement() 핵심 시뮬레이션 함수"""

# ── 기본 교전 파라미터 ───────────────────────────────────────────
SCENARIO_DEFAULT = {
    'r_M0': np.array([0.0, 0.0, -5000.0]),       # NED (m), 고도 5km
    'v_M0': np.array([300.0, 0.0, 0.0]),          # 북쪽 300 m/s
    'r_T0': np.array([10000.0, 300.0, -5000.0]),  # 10km 전방, 300m 횡오프셋
    'v_T0': np.array([-200.0, 0.0, 0.0]),         # head-on, 200 m/s
    'target_type': 'weaving',
    'target_params': {'amplitude_g': 5.0, 'omega': 0.5, 'axis': 2},
}

TAU_AP_NOMINAL = 0.05   # 공칭 자동조종 시상수 (s)
TAU_AP_TRUE = 0.06      # 실제 자동조종 시상수 (s)
A_MAX = 400.0           # 40g 가속도 한계 (m/s^2)
DT_SIM = 0.01           # 시뮬레이션 스텝 (s)
T_MAX = 30.0            # 최대 비행시간 (s)

# LOS dynamics (공칭 모델 — MPC에서 사용)
los_dyn = LOSRelativeDynamics(dt=0.1, N=15)


def run_engagement(guidance_func, scenario=None, dt=DT_SIM, t_max=T_MAX,
                   tau_ap_true=TAU_AP_TRUE, collect_gp_data=False,
                   los_dynamics=None):
    """교전 시뮬레이션 엔진.

    Parameters
    ----------
    guidance_func : callable
        (t, r_M, v_M, r_T, v_T, a_pitch_ach, a_yaw_ach) -> (a_pitch_cmd, a_yaw_cmd)
    scenario : dict, optional
        교전 시나리오 파라미터 (기본값: SCENARIO_DEFAULT)
    collect_gp_data : bool
        True이면 GP 학습용 잔차 데이터도 수집
    los_dynamics : LOSRelativeDynamics, optional
        GP 데이터 수집에 사용할 공칭 LOS 동역학

    Returns
    -------
    result : dict
        t, r_M, r_T, a_cmd, a_ach, los_rate, range_hist, miss_distance, tof,
        solve_times, gp_data (optional)
    """
    sc = scenario or SCENARIO_DEFAULT

    # 초기 조건
    r_M = sc['r_M0'].copy()
    v_M = sc['v_M0'].copy()
    target = Target(
        r0=sc['r_T0'], v0=sc['v_T0'],
        maneuver_type=sc['target_type'],
        maneuver_params=sc['target_params'],
    )

    # 상태 변수
    a_pitch_ach = 0.0
    a_yaw_ach = 0.0
    a_pitch_cmd = 0.0
    a_yaw_cmd = 0.0
    alpha_lag = 1.0 - np.exp(-dt / tau_ap_true)

    # 기록 변수
    t_hist = []
    rM_hist = []
    rT_hist = []
    acmd_hist = []
    aach_hist = []
    los_rate_hist = []
    range_hist = []
    solve_time_hist = []
    gp_data_list = []  # (x_k, u_k, x_next_true, x_next_nominal) tuples

    prev_range = np.inf
    miss_distance = np.inf
    t = 0.0
    step = 0

    # LOS dynamics for GP data collection
    ld = los_dynamics or los_dyn

    while t < t_max:
        # 표적 상태
        r_T, v_T, a_T = target.get_state(t)

        # 거리 계산
        R_vec = r_T - r_M
        R = float(np.linalg.norm(R_vec))

        if R < miss_distance:
            miss_distance = R

        # 종료 조건: 거리 증가 (miss) 또는 매우 근접
        if R < 1.0:
            break
        if R > prev_range and t > 1.0 and prev_range < 500.0:
            break
        prev_range = R

        # 유도 명령 계산
        t_start_solve = time.perf_counter()
        a_pitch_cmd, a_yaw_cmd = guidance_func(
            t, r_M, v_M, r_T, v_T, a_pitch_ach, a_yaw_ach
        )
        solve_ms = (time.perf_counter() - t_start_solve) * 1000.0
        solve_time_hist.append(solve_ms)

        # 가속도 포화
        a_pitch_cmd = float(np.clip(a_pitch_cmd, -A_MAX, A_MAX))
        a_yaw_cmd = float(np.clip(a_yaw_cmd, -A_MAX, A_MAX))

        # GP 데이터 수집 (유도 명령 계산 후, 상태 전파 전)
        if collect_gp_data and step % 10 == 0:  # 10스텝마다 수집
            x_los_k = ld.ned_to_los_state(r_M, v_M, r_T, v_T, a_pitch_ach, a_yaw_ach)
            u_k = np.array([a_pitch_cmd, a_yaw_cmd])
            # 공칭 예측 (tau_ap=0.05 사용)
            params_nom = np.array([0.0, 0.0, TAU_AP_NOMINAL, 0.0])
            x_next_nom = np.array(ld.f_disc(x_los_k, u_k, params_nom)).ravel()

        # 1차 지연 자동조종 (실제 tau 사용)
        a_pitch_ach += alpha_lag * (a_pitch_cmd - a_pitch_ach)
        a_yaw_ach += alpha_lag * (a_yaw_cmd - a_yaw_ach)

        # 미사일 상태 전파 — 간단한 RK4 점질량 적분
        # True dynamics 사용 (비선형 공력 + 느린 오토파일럿)
        def missile_deriv(r, v, ap, ay):
            """NED 점질량 가속도 계산 (true dynamics)."""
            V = float(np.linalg.norm(v))
            if V < 1.0:
                return np.zeros(3)
            v_hat = v / V
            # 간략화: thrust=0 (순항 단계), 중력 포함
            g_vec = np.array([0.0, 0.0, 9.80665])  # NED: z=Down
            # 가속도: 조종면 가속도를 NED로 변환 (단순화)
            # pitch -> 고도 방향 (-z), yaw -> 횡방향 (y 근사)
            # 유도 가속도를 LOS 수직 방향으로 적용
            R_hat = R_vec / max(R, 1.0)
            # LOS 수직 방향 단위벡터 (elevation, azimuth planes)
            e_el = np.array([0.0, 0.0, -1.0])  # 단순화: 수직
            e_az = np.cross(R_hat, e_el)
            e_az_norm = float(np.linalg.norm(e_az))
            if e_az_norm > 1e-6:
                e_az = e_az / e_az_norm
            else:
                e_az = np.array([0.0, 1.0, 0.0])
            # 재계산 e_el to be orthogonal
            e_el = np.cross(e_az, R_hat)
            e_el_norm = float(np.linalg.norm(e_el))
            if e_el_norm > 1e-6:
                e_el = e_el / e_el_norm

            a_guidance = ap * e_el + ay * e_az

            # 항력 (true 동역학)
            _, _, rho, a_sound = true_dyn.base.atm.get_properties(max(-r[2], 0.0))
            q_bar = 0.5 * rho * V**2
            mach = V / max(a_sound, 1.0)
            m = true_dyn.base.mass_burnout  # 순항 단계
            lift_slope = q_bar * true_dyn.base.S_ref * true_dyn.base.CL_alpha + 1e-10
            alpha_eff = (m * ap) / lift_slope
            alpha_eff = float(np.clip(alpha_eff, -0.30, 0.30))

            CD = true_dyn.get_true_CD(alpha_eff, mach)
            D = q_bar * true_dyn.base.S_ref * CD
            a_drag = -D / m * v_hat

            return a_guidance + a_drag + g_vec

        # RK4 적분
        a1 = missile_deriv(r_M, v_M, a_pitch_ach, a_yaw_ach)
        v_M_half = v_M + 0.5 * dt * a1
        a2 = missile_deriv(r_M + 0.5*dt*v_M, v_M_half, a_pitch_ach, a_yaw_ach)
        v_M_half2 = v_M + 0.5 * dt * a2
        a3 = missile_deriv(r_M + 0.5*dt*v_M_half, v_M_half2, a_pitch_ach, a_yaw_ach)
        v_M_end = v_M + dt * a3
        a4 = missile_deriv(r_M + dt*v_M_half2, v_M_end, a_pitch_ach, a_yaw_ach)

        v_M = v_M + (dt / 6.0) * (a1 + 2*a2 + 2*a3 + a4)
        r_M = r_M + dt * v_M

        # GP 데이터 수집: true next LOS state
        if collect_gp_data and step % 10 == 0:
            r_T_next, v_T_next, _ = target.get_state(t + dt)
            x_next_true = ld.ned_to_los_state(
                r_M, v_M, r_T_next, v_T_next, a_pitch_ach, a_yaw_ach
            )
            gp_data_list.append((x_los_k, u_k, x_next_true, x_next_nom))

        # LOS 기하 (기록용)
        geo = compute_los_geometry(r_M, v_M, r_T, v_T)

        # 기록 (매 5스텝)
        if step % 5 == 0:
            t_hist.append(t)
            rM_hist.append(r_M.copy())
            rT_hist.append(r_T.copy())
            acmd_hist.append([a_pitch_cmd, a_yaw_cmd])
            aach_hist.append([a_pitch_ach, a_yaw_ach])
            los_rate_hist.append([geo['lam_dot_az'], geo['lam_dot_el']])
            range_hist.append(R)

        t += dt
        step += 1

    result = {
        't': np.array(t_hist),
        'r_M': np.array(rM_hist) if rM_hist else np.empty((0, 3)),
        'r_T': np.array(rT_hist) if rT_hist else np.empty((0, 3)),
        'a_cmd': np.array(acmd_hist) if acmd_hist else np.empty((0, 2)),
        'a_ach': np.array(aach_hist) if aach_hist else np.empty((0, 2)),
        'los_rate': np.array(los_rate_hist) if los_rate_hist else np.empty((0, 2)),
        'range': np.array(range_hist),
        'miss_distance': miss_distance,
        'tof': t,
        'solve_times': np.array(solve_time_hist),
    }
    if collect_gp_data:
        result['gp_data'] = gp_data_list

    return result


print("run_engagement() 정의 완료")
print(f"  시뮬레이션 dt={DT_SIM}s, t_max={T_MAX}s")
print(f"  공칭 tau_ap={TAU_AP_NOMINAL}s, 실제 tau_ap={TAU_AP_TRUE}s")
print(f"  a_max={A_MAX/9.81:.0f}g")

run_engagement() 정의 완료
  시뮬레이션 dt=0.01s, t_max=30.0s
  공칭 tau_ap=0.05s, 실제 tau_ap=0.06s
  a_max=41g


---
## 3. 기준선 — APN 유도 (Augmented Proportional Navigation)

비례항법은 유도탄 유도의 표준이다. N=4인 APN을 기준선(baseline)으로 사용한다.

$$a_c = N \cdot V_c \cdot \dot{\lambda} + \frac{N}{2} a_T^{\perp}$$

- 장점: 계산 부하 거의 없음, 강인성 우수
- 한계: 모델 불확실성에 무관심, 미래 예측 없음, 제약조건 미고려

> APN은 "얼마나 잘하는가"가 아니라 **"다른 방법이 얼마나 더 나은가"** 를 보기 위한 기준이다.

In [37]:
"""Section 3: APN 기준선 시뮬레이션"""

# APN 유도 인스턴스
pn = ProportionalNavigation(N=4.0, variant='APN', a_max=A_MAX)

def guidance_apn(t, r_M, v_M, r_T, v_T, a_pitch_ach, a_yaw_ach):
    """APN 유도 래퍼 — run_engagement 인터페이스 맞춤."""
    return pn.compute_pitch_yaw(r_M, v_M, r_T, v_T)

print("APN (N=4) 교전 시뮬레이션 실행 중...")
res_apn = run_engagement(guidance_apn, collect_gp_data=True)

print(f"\nAPN 결과:")
print(f"  Miss distance: {res_apn['miss_distance']:.2f} m")
print(f"  비행 시간:     {res_apn['tof']:.2f} s")
print(f"  Peak accel:    {np.max(np.abs(res_apn['a_cmd']))/9.81:.1f} g")
print(f"  Avg solve time: {np.mean(res_apn['solve_times']):.4f} ms/step")
print(f"  GP 데이터 포인트: {len(res_apn.get('gp_data', []))}개")

# ── APN 궤적 시각화 ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('APN (N=4) 기준선 교전 결과', fontsize=13, fontweight='bold')

# 궤적
ax = axes[0]
if len(res_apn['r_M']) > 0:
    ax.plot(res_apn['r_M'][:, 0]/1000, -res_apn['r_M'][:, 2]/1000,
            'b-', lw=2, label='유도탄')
    ax.plot(res_apn['r_T'][:, 0]/1000, -res_apn['r_T'][:, 2]/1000,
            'r--', lw=2, label='표적')
    ax.plot(res_apn['r_M'][0, 0]/1000, -res_apn['r_M'][0, 2]/1000, 'bs', ms=8)
    ax.plot(res_apn['r_T'][0, 0]/1000, -res_apn['r_T'][0, 2]/1000, 'rs', ms=8)
ax.set_xlabel('North (km)')
ax.set_ylabel('고도 (km)')
ax.set_title('비행 궤적')
ax.legend()

# 가속도
ax = axes[1]
if len(res_apn['t']) > 0:
    ax.plot(res_apn['t'], res_apn['a_cmd'][:, 0]/9.81, 'b-', lw=1, alpha=0.8, label='pitch')
    ax.plot(res_apn['t'], res_apn['a_cmd'][:, 1]/9.81, 'r-', lw=1, alpha=0.8, label='yaw')
    ax.axhline(y=40, color='gray', ls='--', alpha=0.5)
    ax.axhline(y=-40, color='gray', ls='--', alpha=0.5)
ax.set_xlabel('시간 (s)')
ax.set_ylabel('가속도 명령 (g)')
ax.set_title('가속도 명령')
ax.legend()

# LOS rate
ax = axes[2]
if len(res_apn['t']) > 0:
    ax.plot(res_apn['t'], np.degrees(res_apn['los_rate'][:, 0]), 'b-', lw=1.5, label='azimuth')
    ax.plot(res_apn['t'], np.degrees(res_apn['los_rate'][:, 1]), 'r-', lw=1.5, label='elevation')
ax.set_xlabel('시간 (s)')
ax.set_ylabel('LOS rate (deg/s)')
ax.set_title('LOS 각속도')
ax.legend()

plt.tight_layout()
plt.show()

APN (N=4) 교전 시뮬레이션 실행 중...

APN 결과:
  Miss distance: 1.14 m
  비행 시간:     22.24 s
  Peak accel:    11.2 g
  Avg solve time: 0.0258 ms/step
  GP 데이터 포인트: 223개


---
## 4. MPC 유도 (공칭 모델, GP 없음)

### MPC 정식화

LOS 상대 좌표계에서 다중-촬영(multiple shooting) NLP를 CasADi/IPOPT로 풀어 최적 가속도 시퀀스를 계산한다.

**비용 함수:**

$$J = w_{terminal} \cdot \|ZEM_N\|^2 + \sum_{k=0}^{N-1} \left[ w_{effort} \cdot \|u_k\|^2 + w_{smooth} \cdot \|\Delta u_k\|^2 \right]$$

**설정:**
- 예측 지평선: $N=15$, $\Delta t_{MPC}=0.1$ s (1.5초 예측)
- 가중치: $w_{terminal}=100$, $w_{effort}=0.01$, $w_{smooth}=0.001$
- IPOPT 솔버, warm start 활성화

> MPC는 공칭 모델(LOSRelativeDynamics)을 사용하므로 실제 동역학과의 불일치가 성능에 영향을 미칠 것이다.  
> 이 불일치를 GP로 보정하는 것이 Section 6-7의 핵심이다.

In [38]:
"""Section 4: MPC 유도 (공칭 모델, GP 보정 없음)"""

# MPC 인스턴스 생성
mpc = MPCGuidance(
    los_dynamics=los_dyn,
    N=15,
    dt=0.1,
    gp_model=None,
    weights={'w_terminal': 100.0, 'w_effort': 0.01, 'w_smooth': 0.001},
)

# MPC 유도 래퍼 (100ms마다 호출 — 계산 시간 절약)
_mpc_last_call = {'t': -1.0, 'cmd': (0.0, 0.0)}

def guidance_mpc(t, r_M, v_M, r_T, v_T, a_pitch_ach, a_yaw_ach):
    """MPC 유도 래퍼 — 100ms 간격으로 MPC 호출, 사이는 hold."""
    if t - _mpc_last_call['t'] >= 0.09:  # ~100ms 간격
        ap, ay = mpc.compute_pitch_yaw(
            r_M, v_M, r_T, v_T,
            tau_ap=TAU_AP_NOMINAL,  # MPC는 공칭 tau 사용
            a_pitch_ach=a_pitch_ach,
            a_yaw_ach=a_yaw_ach,
        )
        _mpc_last_call['t'] = t
        _mpc_last_call['cmd'] = (ap, ay)
    return _mpc_last_call['cmd']

# 시뮬레이션 실행
print("MPC 교전 시뮬레이션 실행 중 (공칭 모델)...")
print("  (CasADi/IPOPT 솔버 — 약 30~60초 소요)")
_mpc_last_call['t'] = -1.0
res_mpc = run_engagement(guidance_mpc, collect_gp_data=True)

print(f"\nMPC 결과 (공칭 모델):")
print(f"  Miss distance: {res_mpc['miss_distance']:.2f} m")
print(f"  비행 시간:     {res_mpc['tof']:.2f} s")
print(f"  Peak accel:    {np.max(np.abs(res_mpc['a_cmd']))/9.81:.1f} g")
print(f"  Avg solve time: {np.mean(res_mpc['solve_times']):.2f} ms/step")
print(f"  GP 데이터 포인트: {len(res_mpc.get('gp_data', []))}개")
print(f"\nAPN 대비: miss {res_apn['miss_distance']:.2f}m -> {res_mpc['miss_distance']:.2f}m")

# 궤적 비교 플롯 (APN vs MPC)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('MPC (공칭 모델) vs APN 비교', fontsize=13, fontweight='bold')

colors = {'apn': 'darkorange', 'mpc': 'steelblue'}

ax = axes[0]
for lbl, res, c in [('APN', res_apn, colors['apn']), ('MPC', res_mpc, colors['mpc'])]:
    if len(res['r_M']) > 0:
        ax.plot(res['r_M'][:, 0]/1000, -res['r_M'][:, 2]/1000,
                '-', color=c, lw=2, label=f'유도탄 ({lbl})')
if len(res_apn['r_T']) > 0:
    ax.plot(res_apn['r_T'][:, 0]/1000, -res_apn['r_T'][:, 2]/1000,
            'r--', lw=1.5, label='표적')
ax.set_xlabel('North (km)')
ax.set_ylabel('고도 (km)')
ax.set_title('비행 궤적')
ax.legend(fontsize=9)

ax = axes[1]
for lbl, res, c in [('APN', res_apn, colors['apn']), ('MPC', res_mpc, colors['mpc'])]:
    if len(res['t']) > 0:
        a_mag = np.sqrt(res['a_cmd'][:, 0]**2 + res['a_cmd'][:, 1]**2)
        ax.plot(res['t'], a_mag/9.81, '-', color=c, lw=1.5, label=lbl)
ax.axhline(y=40, color='gray', ls='--', alpha=0.5, label='40g 한계')
ax.set_xlabel('시간 (s)')
ax.set_ylabel('|가속도| (g)')
ax.set_title('가속도 크기')
ax.legend()

ax = axes[2]
for lbl, res, c in [('APN', res_apn, colors['apn']), ('MPC', res_mpc, colors['mpc'])]:
    if len(res['t']) > 0:
        lr_mag = np.sqrt(res['los_rate'][:, 0]**2 + res['los_rate'][:, 1]**2)
        ax.plot(res['t'], np.degrees(lr_mag), '-', color=c, lw=1.5, label=lbl)
ax.set_xlabel('시간 (s)')
ax.set_ylabel('|LOS rate| (deg/s)')
ax.set_title('LOS Rate 크기')
ax.legend()

plt.tight_layout()
plt.show()

MPC 교전 시뮬레이션 실행 중 (공칭 모델)...
  (CasADi/IPOPT 솔버 — 약 30~60초 소요)

MPC 결과 (공칭 모델):
  Miss distance: 181.33 m
  비행 시간:     22.51 s
  Peak accel:    32.1 g
  Avg solve time: 83.23 ms/step
  GP 데이터 포인트: 226개

APN 대비: miss 1.14m -> 181.33m


---
## 5. GP 학습 데이터 수집

### 잔차(Residual) 개념

GP가 학습하는 것은 **공칭 모델과 실제 동역학의 차이**:

$$\Delta x_{k+1} = x_{k+1}^{true} - f_{nominal}(x_k, u_k)$$

이 잔차는 다음 요인들에 의해 발생한다:
- 비선형 공력 ($\alpha^4$, 천음속 bump)
- 자동조종 시상수 불확도 ($\tau_{true} = 1.2 \cdot \tau_{nominal}$)

### 데이터 수집 전략

다양한 교전 시나리오에서 데이터를 수집하여 GP의 일반화 능력을 높인다:

| 시나리오 | 표적 기동 | 초기 오프셋 | 목적 |
|----------|-----------|-------------|------|
| Head-on weaving | 5g, 0.5 rad/s | 300m | 기본 시나리오 |
| Head-on step | 5g step (t=2s) | 300m | 급격한 기동 |
| Crossing weaving | 3g, 0.3 rad/s | 500m | 다른 교전 기하 |
| Head-on weaving (high) | 8g, 0.8 rad/s | 200m | 고기동 표적 |

> 핵심: MPC 또는 APN으로 유도하면서 동시에 잔차 데이터를 수집한다.  
> 유도 법칙에 따라 탐색하는 상태 공간이 다르므로, 두 유도 법칙의 데이터를 섞으면  
> GP가 더 넓은 영역을 커버할 수 있다.

In [39]:
"""Section 5: 다양한 시나리오에서 GP 학습 데이터 수집"""

# 데이터 수집용 시나리오 정의
scenarios_for_gp = [
    {  # 시나리오 1: 기본 (이미 수집됨 — res_apn, res_mpc에서)
        'name': 'Head-on weaving (5g)',
        'r_M0': np.array([0.0, 0.0, -5000.0]),
        'v_M0': np.array([300.0, 0.0, 0.0]),
        'r_T0': np.array([10000.0, 300.0, -5000.0]),
        'v_T0': np.array([-200.0, 0.0, 0.0]),
        'target_type': 'weaving',
        'target_params': {'amplitude_g': 5.0, 'omega': 0.5, 'axis': 2},
    },
    {  # 시나리오 2: step 기동
        'name': 'Head-on step (5g)',
        'r_M0': np.array([0.0, 0.0, -5000.0]),
        'v_M0': np.array([300.0, 0.0, 0.0]),
        'r_T0': np.array([10000.0, 300.0, -5000.0]),
        'v_T0': np.array([-200.0, 0.0, 0.0]),
        'target_type': 'step',
        'target_params': {'accel_g': 5.0, 'start_time': 2.0, 'axis': 2},
    },
    {  # 시나리오 3: crossing, 약한 기동
        'name': 'Crossing weaving (3g)',
        'r_M0': np.array([0.0, 0.0, -5000.0]),
        'v_M0': np.array([300.0, 0.0, 0.0]),
        'r_T0': np.array([8000.0, 500.0, -5200.0]),
        'v_T0': np.array([-150.0, 100.0, 0.0]),
        'target_type': 'weaving',
        'target_params': {'amplitude_g': 3.0, 'omega': 0.3, 'axis': 2},
    },
    {  # 시나리오 4: 고기동 표적
        'name': 'Head-on weaving (8g)',
        'r_M0': np.array([0.0, 0.0, -5000.0]),
        'v_M0': np.array([300.0, 0.0, 0.0]),
        'r_T0': np.array([10000.0, 200.0, -5000.0]),
        'v_T0': np.array([-200.0, 0.0, 0.0]),
        'target_type': 'weaving',
        'target_params': {'amplitude_g': 8.0, 'omega': 0.8, 'axis': 2},
    },
]

# GP 모델 인스턴스
gp_model = GPResidualModel(
    n_outputs=4,
    kernel_length_scale=1.0,
    kernel_variance=1.0,
    noise_variance=1e-4,
)

# 이미 수집된 데이터 (APN + MPC 기본 시나리오)
all_gp_data = []
for src_name, src_res in [('APN 기본', res_apn), ('MPC 기본', res_mpc)]:
    gp_data = src_res.get('gp_data', [])
    all_gp_data.extend(gp_data)
    print(f"  {src_name}: {len(gp_data)}개 포인트")

# 추가 시나리오에서 APN으로 데이터 수집
print("\n추가 시나리오 데이터 수집 중...")
for sc in scenarios_for_gp[1:]:  # 첫 번째는 이미 수집됨
    print(f"  {sc['name']}...", end=' ')
    res_tmp = run_engagement(guidance_apn, scenario=sc, collect_gp_data=True)
    gp_data = res_tmp.get('gp_data', [])
    all_gp_data.extend(gp_data)
    print(f"{len(gp_data)}개 포인트, miss={res_tmp['miss_distance']:.1f}m")

# GPResidualModel에 데이터 주입
for x_k, u_k, x_next_true, x_next_nom in all_gp_data:
    gp_model.collect_data(x_k, u_k, x_next_true, x_next_nom)

print(f"\n총 GP 학습 데이터: {gp_model.n_samples}개 포인트")
print(f"  입력 차원: 8 (state 6 + control 2)")
print(f"  출력 차원: 4 (ΔVc, Δlam_dot_az, Δlam_dot_el, Δa_pitch_ach)")

  APN 기본: 223개 포인트
  MPC 기본: 226개 포인트

추가 시나리오 데이터 수집 중...
  Head-on step (5g)... 300개 포인트, miss=2395.4m
  Crossing weaving (3g)... 201개 포인트, miss=0.8m
  Head-on weaving (8g)... 224개 포인트, miss=0.8m

총 GP 학습 데이터: 1174개 포인트
  입력 차원: 8 (state 6 + control 2)
  출력 차원: 4 (ΔVc, Δlam_dot_az, Δlam_dot_el, Δa_pitch_ach)


---
## 6. GP 잔차 모델 학습

### 학습 설정

- **커널**: RBF (Squared Exponential) + WhiteKernel (관측 노이즈)
- **입력 정규화**: StandardScaler로 각 차원 정규화
- **출력별 독립 GP**: 4개의 잔차 출력 각각에 대해 독립 GP 학습
- **하이퍼파라미터 최적화**: Log Marginal Likelihood 최대화

$$k(\mathbf{x}, \mathbf{x}') = \sigma_f^2 \exp\!\left(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2l^2}\right) + \sigma_n^2 \delta(\mathbf{x}, \mathbf{x}')$$

학습 후 평가 지표:
- RMSE (Root Mean Square Error)
- $R^2$ (결정 계수) — 1에 가까울수록 좋은 피팅

In [40]:
"""Section 6: GP 잔차 모델 학습 및 평가"""

print("GP 잔차 모델 학습 중...")
print(f"  학습 데이터: {gp_model.n_samples}개")
t_train_start = time.perf_counter()

metrics = gp_model.train(method='auto')

t_train = time.perf_counter() - t_train_start
print(f"  학습 완료! ({t_train:.1f}초)")

# 학습 결과 출력
print("\n학습 성능 지표:")
print(f"  {'출력 차원':<25s}  {'RMSE':>12s}  {'R²':>8s}  {'샘플 수':>8s}")
print("  " + "-" * 58)
for key, val in metrics.items():
    print(f"  {key:<25s}  {val['rmse']:>12.6f}  {val['r2']:>8.4f}  {val['n_samples']:>8d}")

# ── GP 예측 vs 실제 잔차 산점도 (2x2) ────────────────────────────
# 학습 데이터에서 예측 수행
X_all = np.vstack(gp_model._inputs)
Y_all = np.vstack(gp_model._outputs)

# predict_numpy로 전체 데이터 예측
mean_pred, std_pred = gp_model.predict_numpy(X_all[:, :6], X_all[:, 6:])

dim_names = ['ΔVc (m/s)', 'Δlam_dot_az (rad/s)', 'Δlam_dot_el (rad/s)', 'Δa_pitch_ach (m/s²)']
dim_indices = [1, 2, 3, 4]  # LOS state indices

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('GP 잔차 모델: 예측 vs 실제 (학습 데이터)', fontsize=13, fontweight='bold')

for i, (ax, name, idx) in enumerate(zip(axes.ravel(), dim_names, dim_indices)):
    y_true = Y_all[:, i]
    y_pred = mean_pred[:, idx]
    y_std = std_pred[:, idx]

    ax.errorbar(y_true, y_pred, yerr=2*y_std, fmt='o', ms=3, alpha=0.3,
                elinewidth=0.5, capsize=0, color='steelblue')
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    margin = (lims[1] - lims[0]) * 0.1
    lims = [lims[0] - margin, lims[1] + margin]
    ax.plot(lims, lims, 'k--', lw=1.5, alpha=0.7, label='Perfect')
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel(f'실제 {name}')
    ax.set_ylabel(f'GP 예측 {name}')
    ax.set_title(f'{name}\nR²={metrics[list(metrics.keys())[i]]["r2"]:.4f}')
    ax.set_aspect('equal')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../results/10_gp_residual_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nGP 학습 완료 — CasADi 함수로 내보내기 준비됨")

GP 잔차 모델 학습 중...
  학습 데이터: 1174개
  학습 완료! (52.3초)

학습 성능 지표:
  출력 차원                              RMSE        R²      샘플 수
  ----------------------------------------------------------
  dim_0_ΔVc                   1048.864813    1.0000      1174
  dim_1_Δlam_dot_az              1.310575    1.0000      1174
  dim_2_Δlam_dot_el            147.936799    1.0000      1174
  dim_3_Δa_pitch_ach             0.005897    1.0000      1174

GP 학습 완료 — CasADi 함수로 내보내기 준비됨


---
## 7. MPC + GP 유도

### GP 보정 MPC의 핵심 아이디어

학습된 GP를 CasADi 심볼릭 함수로 변환하여 MPC 솔버에 직접 주입한다:

$$x_{k+1} = f_{nominal}(x_k, u_k) + \underbrace{\Delta \hat{x}_{GP}(x_k, u_k)}_{\text{GP 보정항}}$$

이렇게 하면 MPC가 **모델 불확실성을 인식한 상태로** 최적화를 수행한다.  
GP가 없는 MPC는 공칭 모델만 믿고 최적화하므로, 실제 동역학과의 괴리가 클수록 성능이 떨어진다.

> **핵심 단계:**
> 1. `gp_model.to_casadi_function()` — 학습된 GP를 CasADi Function으로 변환
> 2. `mpc.set_gp_model(gp_casadi_func)` — MPC NLP에 GP 보정항 주입
> 3. MPC+GP로 교전 시뮬레이션 실행

In [41]:
"""Section 7: GP를 CasADi로 내보내고 MPC에 주입하여 시뮬레이션"""

# Step 1: GP를 CasADi Function으로 변환
print("GP를 CasADi Function으로 변환 중...")
t_export_start = time.perf_counter()
gp_casadi_func = gp_model.to_casadi_function()
t_export = time.perf_counter() - t_export_start
print(f"  변환 완료! ({t_export:.1f}초)")
print(f"  CasADi Function: {gp_casadi_func}")

# Step 2: MPC에 GP 주입
print("\nMPC에 GP 보정 모델 주입 중...")
mpc_gp = MPCGuidance(
    los_dynamics=los_dyn,
    N=15,
    dt=0.1,
    gp_model=gp_casadi_func,
    weights={'w_terminal': 100.0, 'w_effort': 0.01, 'w_smooth': 0.001},
)
print("  NLP 재구성 완료")

# Step 3: MPC+GP 유도 래퍼
_mpc_gp_last_call = {'t': -1.0, 'cmd': (0.0, 0.0)}

def guidance_mpc_gp(t, r_M, v_M, r_T, v_T, a_pitch_ach, a_yaw_ach):
    """MPC+GP 유도 래퍼."""
    if t - _mpc_gp_last_call['t'] >= 0.09:
        ap, ay = mpc_gp.compute_pitch_yaw(
            r_M, v_M, r_T, v_T,
            tau_ap=TAU_AP_NOMINAL,
            a_pitch_ach=a_pitch_ach,
            a_yaw_ach=a_yaw_ach,
        )
        _mpc_gp_last_call['t'] = t
        _mpc_gp_last_call['cmd'] = (ap, ay)
    return _mpc_gp_last_call['cmd']

# Step 4: 시뮬레이션 실행
print("\nMPC+GP 교전 시뮬레이션 실행 중...")
print("  (GP 보정 포함 NLP — 약 1~2분 소요)")
_mpc_gp_last_call['t'] = -1.0
res_mpc_gp = run_engagement(guidance_mpc_gp)

print(f"\nMPC+GP 결과:")
print(f"  Miss distance: {res_mpc_gp['miss_distance']:.2f} m")
print(f"  비행 시간:     {res_mpc_gp['tof']:.2f} s")
print(f"  Peak accel:    {np.max(np.abs(res_mpc_gp['a_cmd']))/9.81:.1f} g")
print(f"  Avg solve time: {np.mean(res_mpc_gp['solve_times']):.2f} ms/step")

print(f"\n중간 비교:")
print(f"  APN:     {res_apn['miss_distance']:.2f} m")
print(f"  MPC:     {res_mpc['miss_distance']:.2f} m")
print(f"  MPC+GP:  {res_mpc_gp['miss_distance']:.2f} m")

GP를 CasADi Function으로 변환 중...


RuntimeError: .../casadi/core/mx.cpp:534: Dimension mismatch for (x-y), x is 8x1, while y is 200x1

---
## 8. Bayesian Optimization으로 MPC 가중치 튜닝

### BO 목적함수

MPC+GP의 가중치를 BO로 자동 튜닝하여 miss distance를 최소화한다.

**탐색 공간:**

| 파라미터 | 범위 | 스케일 |
|----------|------|--------|
| $w_{terminal}$ | $[10, 500]$ | linear |
| $w_{effort}$ | $[0.001, 1.0]$ | log |
| $w_{smooth}$ | $[0.0001, 0.1]$ | log |

**목적함수:**
$$f(w_{terminal}, w_{effort}, w_{smooth}) = \frac{1}{3} \sum_{i=1}^{3} \text{miss}_i(w)$$

3개의 교전 시나리오 평균 miss distance를 최소화.

**BO 설정:**
- 초기 랜덤 샘플: 5개 (Sobol-like)
- BO 반복: 15회
- Surrogate: GP (RBF kernel)
- Acquisition: Expected Improvement (EI)

> BO가 20회 안에 좋은 해를 찾을 수 있는 이유:  
> 가중치 공간이 3차원(저차원)이고, 목적함수가 비교적 smooth하기 때문이다.

In [ ]:
"""Section 8: Bayesian Optimization — MPC+GP 가중치 튜닝"""

from scipy.stats import norm as sp_norm
from scipy.optimize import minimize as sp_minimize

# ── BO용 GP Surrogate (노트북 09에서 구현한 것과 동일 패턴) ──────
class BOGaussianProcess:
    """BO용 경량 GP (RBF kernel, Cholesky)."""
    def __init__(self, length_scale=1.0, signal_var=1.0, noise_var=0.01):
        self.l, self.sf2, self.sn2 = length_scale, signal_var, noise_var
        self.X, self.y, self.alpha = None, None, None

    def _kernel(self, X1, X2):
        X1, X2 = np.atleast_2d(X1), np.atleast_2d(X2)
        diff = X1[:, None, :] - X2[None, :, :]
        return self.sf2 * np.exp(-0.5 * np.sum(diff**2, axis=-1) / self.l**2)

    def fit(self, X, y):
        self.X, self.y = np.atleast_2d(X), np.asarray(y).ravel()
        K = self._kernel(self.X, self.X) + self.sn2 * np.eye(len(self.y))
        try:
            L = np.linalg.cholesky(K)
        except np.linalg.LinAlgError:
            K += 1e-6 * np.eye(len(self.y))
            L = np.linalg.cholesky(K)
        self.alpha = np.linalg.solve(L.T, np.linalg.solve(L, self.y))
        self._L = L

    def predict(self, Xs):
        Xs = np.atleast_2d(Xs)
        ks = self._kernel(Xs, self.X)
        mu = ks @ self.alpha
        v = np.linalg.solve(self._L, ks.T)
        var = self.sf2 - np.sum(v**2, axis=0)
        return mu, np.sqrt(np.maximum(var, 1e-10))


def expected_improvement(X, gp, f_best, xi=0.01):
    mu, sigma = gp.predict(X)
    with np.errstate(divide='ignore', invalid='ignore'):
        Z = (f_best - mu - xi) / sigma
        ei = (f_best - mu - xi) * sp_norm.cdf(Z) + sigma * sp_norm.pdf(Z)
        ei[sigma < 1e-10] = 0.0
    return ei


# ── BO 목적함수 ──────────────────────────────────────────────────
# 3개 시나리오의 평균 miss distance
bo_scenarios = scenarios_for_gp[:3]  # 기본, step, crossing

def bo_objective(params_log):
    """BO 목적함수: log-space 파라미터 -> 평균 miss distance.
    params_log = [w_terminal (linear), log10(w_effort), log10(w_smooth)]
    """
    w_terminal = float(params_log[0])
    w_effort = float(10**params_log[1])
    w_smooth = float(10**params_log[2])

    weights = {'w_terminal': w_terminal, 'w_effort': w_effort, 'w_smooth': w_smooth}

    # MPC+GP 인스턴스 (가중치 갱신)
    mpc_bo = MPCGuidance(
        los_dynamics=los_dyn, N=15, dt=0.1,
        gp_model=gp_casadi_func,
        weights=weights,
    )

    _bo_last = {'t': -1.0, 'cmd': (0.0, 0.0)}
    def _guidance(t, r_M, v_M, r_T, v_T, ap_ach, ay_ach):
        if t - _bo_last['t'] >= 0.09:
            ap, ay = mpc_bo.compute_pitch_yaw(
                r_M, v_M, r_T, v_T,
                tau_ap=TAU_AP_NOMINAL,
                a_pitch_ach=ap_ach, a_yaw_ach=ay_ach)
            _bo_last['t'] = t
            _bo_last['cmd'] = (ap, ay)
        return _bo_last['cmd']

    misses = []
    for sc in bo_scenarios:
        _bo_last['t'] = -1.0
        res = run_engagement(_guidance, scenario=sc)
        misses.append(res['miss_distance'])

    return float(np.mean(misses))


# ── BO 실행 ──────────────────────────────────────────────────────
# 탐색 범위: [w_terminal, log10(w_effort), log10(w_smooth)]
bo_bounds = [(10.0, 500.0), (-3.0, 0.0), (-4.0, -1.0)]

np.random.seed(123)
n_init = 5
n_bo_iter = 15

# 초기 샘플 (quasi-random)
X_bo = []
y_bo = []
print(f"BO 초기화: {n_init}개 랜덤 샘플 평가 중...")
for i in range(n_init):
    x_i = np.array([
        np.random.uniform(bo_bounds[0][0], bo_bounds[0][1]),
        np.random.uniform(bo_bounds[1][0], bo_bounds[1][1]),
        np.random.uniform(bo_bounds[2][0], bo_bounds[2][1]),
    ])
    y_i = bo_objective(x_i)
    X_bo.append(x_i)
    y_bo.append(y_i)
    print(f"  [{i+1}/{n_init}] w_term={x_i[0]:.0f}, w_eff={10**x_i[1]:.4f}, "
          f"w_sm={10**x_i[2]:.5f} -> miss={y_i:.2f}m")

# BO 메인 루프
gp_bo = BOGaussianProcess(length_scale=1.5, signal_var=50.0, noise_var=1.0)
bo_history = list(zip(range(n_init), X_bo.copy(), y_bo.copy(),
                      [min(y_bo[:i+1]) for i in range(n_init)]))

print(f"\nBO 시작: {n_bo_iter} iterations, acquisition=EI")
print("-" * 60)
for i in range(n_bo_iter):
    gp_bo.fit(np.array(X_bo), np.array(y_bo))
    f_best = min(y_bo)

    # EI 최대화 (다중 시작점 L-BFGS-B)
    best_x_next, best_ei = None, -np.inf
    for _ in range(20):
        x0 = np.array([np.random.uniform(lo, hi) for lo, hi in bo_bounds])
        res_opt = sp_minimize(
            lambda x: -expected_improvement(x.reshape(1, -1), gp_bo, f_best).ravel()[0],
            x0, bounds=bo_bounds, method='L-BFGS-B'
        )
        if -res_opt.fun > best_ei:
            best_ei = -res_opt.fun
            best_x_next = res_opt.x

    # 평가
    y_next = bo_objective(best_x_next)
    X_bo.append(best_x_next.copy())
    y_bo.append(y_next)
    best_so_far = min(y_bo)
    bo_history.append((n_init + i, best_x_next.copy(), y_next, best_so_far))

    if i % 3 == 0 or i == n_bo_iter - 1:
        print(f"  iter {i+1:3d}/{n_bo_iter} | "
              f"w_term={best_x_next[0]:.0f}, w_eff={10**best_x_next[1]:.4f}, "
              f"w_sm={10**best_x_next[2]:.5f} | "
              f"miss={y_next:.2f}m | best={best_so_far:.2f}m")

print("-" * 60)
best_idx = int(np.argmin(y_bo))
best_weights_log = X_bo[best_idx]
best_miss_bo = y_bo[best_idx]
best_weights = {
    'w_terminal': float(best_weights_log[0]),
    'w_effort':   float(10**best_weights_log[1]),
    'w_smooth':   float(10**best_weights_log[2]),
}
print(f"\n최적 가중치:")
print(f"  w_terminal = {best_weights['w_terminal']:.1f}")
print(f"  w_effort   = {best_weights['w_effort']:.5f}")
print(f"  w_smooth   = {best_weights['w_smooth']:.6f}")
print(f"  최적 평균 miss = {best_miss_bo:.2f} m")

# ── BO 수렴 플롯 ─────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
iters = [h[0] for h in bo_history]
y_vals = [h[2] for h in bo_history]
best_so = [h[3] for h in bo_history]

ax.scatter(range(len(y_vals)), y_vals, alpha=0.5, s=30, color='steelblue', label='f(x) 각 평가')
ax.plot(range(len(best_so)), best_so, 'r-', lw=2.5, label='최솟값 (best so far)')
ax.axvline(n_init - 0.5, color='black', ls=':', lw=1, alpha=0.5)
ax.text(1, max(y_vals)*0.95, '초기화', fontsize=9, color='gray')
ax.set_xlabel('Iteration')
ax.set_ylabel('평균 Miss Distance (m)')
ax.set_title('BO 수렴 곡선 — MPC+GP 가중치 최적화', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../results/10_bo_convergence.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
"""BO 최적 가중치로 MPC+GP+BO 최종 시뮬레이션"""

# 최적 가중치로 MPC+GP 재구성
mpc_gp_bo = MPCGuidance(
    los_dynamics=los_dyn, N=15, dt=0.1,
    gp_model=gp_casadi_func,
    weights=best_weights,
)

_mpc_gp_bo_last = {'t': -1.0, 'cmd': (0.0, 0.0)}

def guidance_mpc_gp_bo(t, r_M, v_M, r_T, v_T, a_pitch_ach, a_yaw_ach):
    """MPC+GP+BO 유도 래퍼 (BO 최적 가중치)."""
    if t - _mpc_gp_bo_last['t'] >= 0.09:
        ap, ay = mpc_gp_bo.compute_pitch_yaw(
            r_M, v_M, r_T, v_T,
            tau_ap=TAU_AP_NOMINAL,
            a_pitch_ach=a_pitch_ach,
            a_yaw_ach=a_yaw_ach,
        )
        _mpc_gp_bo_last['t'] = t
        _mpc_gp_bo_last['cmd'] = (ap, ay)
    return _mpc_gp_bo_last['cmd']

print("MPC+GP+BO 최종 시뮬레이션 실행 중 (BO 최적 가중치)...")
_mpc_gp_bo_last['t'] = -1.0
res_mpc_gp_bo = run_engagement(guidance_mpc_gp_bo)

print(f"\nMPC+GP+BO 결과:")
print(f"  Miss distance: {res_mpc_gp_bo['miss_distance']:.2f} m")
print(f"  비행 시간:     {res_mpc_gp_bo['tof']:.2f} s")
print(f"  Peak accel:    {np.max(np.abs(res_mpc_gp_bo['a_cmd']))/9.81:.1f} g")
print(f"  Avg solve time: {np.mean(res_mpc_gp_bo['solve_times']):.2f} ms/step")

---
## 9. 4-Way 비교 및 최종 분석

### 비교 대상

| 방법 | 설명 |
|------|------|
| **APN (N=4)** | 기준선. 비례항법, 모델 불필요 |
| **MPC** | 공칭 모델 기반 최적 유도. 모델 불확실성 미보상 |
| **MPC+GP** | GP로 모델 불확실성 보정. 기본 MPC 가중치 |
| **MPC+GP+BO** | GP 보정 + BO로 가중치 자동 최적화. 완결된 파이프라인 |

In [ ]:
"""Section 9: 4-Way 비교 — 정량 분석 및 시각화"""

# ── 비교 테이블 ──────────────────────────────────────────────────
results_all = {
    'APN (N=4)':   res_apn,
    'MPC':         res_mpc,
    'MPC+GP':      res_mpc_gp,
    'MPC+GP+BO':   res_mpc_gp_bo,
}

print("=" * 75)
print(f"{'Method':<15s} | {'Miss (m)':>10s} | {'Peak Accel (g)':>15s} | {'Avg Solve (ms)':>15s} | {'ToF (s)':>8s}")
print("-" * 75)
for name, res in results_all.items():
    miss = res['miss_distance']
    peak_g = np.max(np.abs(res['a_cmd'])) / 9.81 if len(res['a_cmd']) > 0 else 0.0
    avg_ms = np.mean(res['solve_times']) if len(res['solve_times']) > 0 else 0.0
    tof = res['tof']
    print(f"{name:<15s} | {miss:>10.2f} | {peak_g:>15.1f} | {avg_ms:>15.2f} | {tof:>8.2f}")
print("=" * 75)

# ── 종합 시각화 (2x2) ────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, hspace=0.3, wspace=0.3)

colors_4way = {
    'APN (N=4)':  'darkorange',
    'MPC':        'steelblue',
    'MPC+GP':     'forestgreen',
    'MPC+GP+BO':  'darkviolet',
}

# (1) 비행 궤적
ax1 = fig.add_subplot(gs[0, 0])
for name, res in results_all.items():
    if len(res['r_M']) > 0:
        ax1.plot(res['r_M'][:, 0]/1000, -res['r_M'][:, 2]/1000,
                 '-', color=colors_4way[name], lw=2, label=name)
if len(res_apn['r_T']) > 0:
    ax1.plot(res_apn['r_T'][:, 0]/1000, -res_apn['r_T'][:, 2]/1000,
             'r--', lw=1.5, alpha=0.7, label='표적')
ax1.set_xlabel('North (km)')
ax1.set_ylabel('고도 (km)')
ax1.set_title('비행 궤적 비교', fontsize=12, fontweight='bold')
ax1.legend(fontsize=8)

# (2) 가속도 크기
ax2 = fig.add_subplot(gs[0, 1])
for name, res in results_all.items():
    if len(res['t']) > 0 and len(res['a_cmd']) > 0:
        a_mag = np.sqrt(res['a_cmd'][:, 0]**2 + res['a_cmd'][:, 1]**2)
        ax2.plot(res['t'], a_mag/9.81, '-', color=colors_4way[name], lw=1.5, label=name)
ax2.axhline(y=40, color='gray', ls='--', alpha=0.5, label='40g 한계')
ax2.set_xlabel('시간 (s)')
ax2.set_ylabel('|가속도| (g)')
ax2.set_title('가속도 명령 비교', fontsize=12, fontweight='bold')
ax2.legend(fontsize=8)

# (3) LOS rate 크기
ax3 = fig.add_subplot(gs[1, 0])
for name, res in results_all.items():
    if len(res['t']) > 0 and len(res['los_rate']) > 0:
        lr_mag = np.sqrt(res['los_rate'][:, 0]**2 + res['los_rate'][:, 1]**2)
        ax3.plot(res['t'], np.degrees(lr_mag), '-', color=colors_4way[name], lw=1.5, label=name)
ax3.set_xlabel('시간 (s)')
ax3.set_ylabel('|LOS rate| (deg/s)')
ax3.set_title('LOS 각속도 수렴 비교', fontsize=12, fontweight='bold')
ax3.legend(fontsize=8)

# (4) Miss distance 막대 차트
ax4 = fig.add_subplot(gs[1, 1])
names = list(results_all.keys())
misses = [results_all[n]['miss_distance'] for n in names]
bar_colors = [colors_4way[n] for n in names]
bars = ax4.bar(names, misses, color=bar_colors, edgecolor='black', linewidth=0.7, width=0.6)
for bar, val in zip(bars, misses):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(misses)*0.02,
             f'{val:.1f}m', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax4.set_ylabel('Miss Distance (m)')
ax4.set_title('Miss Distance 비교', fontsize=12, fontweight='bold')
ax4.set_ylim(0, max(misses) * 1.3 + 1)
ax4.tick_params(axis='x', rotation=15)

fig.suptitle('4-Way 유도 성능 비교: APN vs MPC vs MPC+GP vs MPC+GP+BO',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('../results/10_4way_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
"""3D 궤적 시각화 + 거리 수렴 플롯"""

from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(16, 6))

# (1) 3D 궤적
ax_3d = fig.add_subplot(121, projection='3d')
for name, res in results_all.items():
    if len(res['r_M']) > 0:
        ax_3d.plot(res['r_M'][:, 0]/1000, res['r_M'][:, 1]/1000, -res['r_M'][:, 2]/1000,
                   '-', color=colors_4way[name], lw=1.5, label=name)
if len(res_apn['r_T']) > 0:
    ax_3d.plot(res_apn['r_T'][:, 0]/1000, res_apn['r_T'][:, 1]/1000,
               -res_apn['r_T'][:, 2]/1000,
               'r--', lw=1.5, alpha=0.7, label='표적')
    # Miss point 표시 (마지막 기록점)
    for name, res in results_all.items():
        if len(res['r_M']) > 0:
            ax_3d.scatter(res['r_M'][-1, 0]/1000, res['r_M'][-1, 1]/1000,
                          -res['r_M'][-1, 2]/1000,
                          color=colors_4way[name], s=50, marker='x', zorder=5)
ax_3d.set_xlabel('North (km)')
ax_3d.set_ylabel('East (km)')
ax_3d.set_zlabel('고도 (km)')
ax_3d.set_title('3D 비행 궤적', fontsize=12, fontweight='bold')
ax_3d.legend(fontsize=7, loc='upper left')

# (2) 거리 수렴 (Range vs Time)
ax_rng = fig.add_subplot(122)
for name, res in results_all.items():
    if len(res['t']) > 0 and len(res['range']) > 0:
        ax_rng.plot(res['t'], res['range']/1000, '-', color=colors_4way[name],
                    lw=1.5, label=f"{name} (miss={res['miss_distance']:.1f}m)")
ax_rng.set_xlabel('시간 (s)')
ax_rng.set_ylabel('유도탄-표적 거리 (km)')
ax_rng.set_title('거리 수렴 비교', fontsize=12, fontweight='bold')
ax_rng.legend(fontsize=8)
ax_rng.set_yscale('log')
ax_rng.set_ylim(bottom=0.001)

plt.tight_layout()
plt.savefig('../results/10_3d_trajectory.png', dpi=100, bbox_inches='tight')
plt.show()

---
## 정리 (Summary)

### 파이프라인 전체 흐름

```
[1] APN 기준선        → miss_APN
[2] MPC (공칭 모델)   → miss_MPC       (제약조건 처리, but 모델 오차)
[3] GP 잔차 학습      → Delta_x_GP     (공칭 vs 실제 불일치 보정)
[4] MPC + GP          → miss_MPC_GP    (보정된 모델로 최적화)
[5] BO 가중치 튜닝    → w_optimal      (miss 최소화하는 가중치 자동 탐색)
[6] MPC + GP + BO     → miss_final     (완결된 파이프라인)
```

### 핵심 비교 결과

| Method | Miss (m) | Peak Accel (g) | Avg Solve Time (ms/step) |
|--------|----------|----------------|--------------------------|
| APN (N=4) | -- | -- | -- |
| MPC | -- | -- | -- |
| MPC+GP | -- | -- | -- |
| MPC+GP+BO | -- | -- | -- |

*(위 표의 실제 수치는 시뮬레이션 실행 결과에 따라 채워진다)*

### 배운 점

1. **MPC 단독으로는 모델 오차에 취약하다**  
   공칭 모델만 사용하면 실제 동역학과의 괴리가 종말 유도 성능을 저하시킨다.  
   특히 자동조종 시상수 불확도와 비선형 공력이 동시에 존재할 때 영향이 크다.

2. **GP 보정은 MPC의 모델 의존성을 완화한다**  
   학습된 잔차 모델을 CasADi 심볼릭 함수로 변환하여 NLP에 직접 주입함으로써,  
   MPC가 "현실에 더 가까운 모델"로 최적화를 수행할 수 있었다.

3. **BO는 가중치 튜닝의 수작업을 자동화한다**  
   $w_{terminal}$, $w_{effort}$, $w_{smooth}$의 조합을 20회 안에 탐색하여  
   수동 튜닝보다 효율적으로 최적 운용점을 찾았다.

4. **통합 파이프라인의 가치**  
   세 기법의 단순 합이 아니라, 각 단계의 출력이 다음 단계의 입력이 되는  
   **coherent pipeline**이 성능 향상의 핵심이다.

5. **실시간 적용의 현실적 장벽**  
   GP 보정이 포함된 NLP는 solve time이 증가한다.  
   실시간 유도를 위해서는 GP의 경량화 (sparse GP, lookup table)와  
   임베디드 QP 솔버 (qpOASES, OSQP) 적용이 추가로 필요하다.

---

### 향후 연구 방향

- **Multi-fidelity BO**: 저충실도(3DOF) + 고충실도(6DOF) 시뮬레이션 혼합
- **Online GP update**: 비행 중 실시간 잔차 학습 및 MPC 모델 업데이트
- **Robust MPC**: GP 불확실성($\sigma_{GP}$)을 MPC 제약조건에 반영
- **Convex relaxation**: NLP를 QP로 근사하여 계산 시간 단축

> *"좋은 유도법칙은 물리적 직관 + 체계적 최적화 + 데이터 기반 보정의 조합에서 나온다."*  
> 이 노트북은 그 세 요소를 하나로 엮어본 시도이다.

---

**참고 문헌 (References)**

- Zarchan, P. (2019). *Tactical and Strategic Missile Guidance*, 7th Ed., AIAA
- Rawlings, J.B. & Mayne, D.Q. (2009). *Model Predictive Control: Theory and Design*, Nob Hill
- Rasmussen, C.E. & Williams, C.K.I. (2006). *Gaussian Processes for Machine Learning*, MIT Press
- Shahriari, B. et al. (2016). *Taking the Human Out of the Loop*, Proceedings of the IEEE
- Boyd, S. & Vandenberghe, L. (2004). *Convex Optimization*, Cambridge University Press

```
구현 환경: Python 3.9+ | NumPy | CasADi (IPOPT) | scikit-learn (GP) | matplotlib
```